# Direct vs Indirect Genetic Effects in `xftsim`

A walkthrough of how to:

1. Install the `ap_indirect` development branch.
2. Build family-based simulations of increasing complexity using the new
   formula DSL — DGE-only, two correlated traits, per-haplotype
   transmitted scores, vertical (indirect) transmission with a
   variance-pinned IGE channel.
3. Verify each stage's variance / covariance / identity targets.
4. Estimate direct and indirect coefficients post-hoc from the simulated
   trios via Kong-style non-transmitted-coefficient (NTC) regression.
5. Layer in assortative mating with single- and cross-trait targets, and
   see what happens when the per-cell exchangeable correlation becomes
   infeasible.

The architecture decisions made here (sourcing IGE from the parent's
*full* DGE for biological realism while separately tracking the
non-transmitted parental allele score for clean identification) are
explained inline as we build the formula up.

## 0. Installation

The `ap_indirect` branch is on top of the `ajay` rewrite — numpy-backed
data structures, a formula DSL for architectures, and a new simulation
loop. None of the legacy xarray-based components are needed for the
work in this notebook.

### Option A — clone and use the dev script

```bash
git clone -b ap_indirect https://github.com/border-lab/xftsim.git
cd xftsim
./scripts/setup-dev.sh        # creates .venv from requirements-lock.txt
source .venv/bin/activate
```

### Option B — pip install directly into an existing env

```bash
pip install "git+https://github.com/border-lab/xftsim.git@ap_indirect#egg=xftsim[legacy,dev]"
```

The `[legacy]` extra pulls in `sgkit` (only needed if you load real zarr
genomes — not used here). `[dev]` pulls in pytest etc.

### Smoke check

In [1]:
import xftsim
import numpy as np, pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

print("xftsim:", xftsim.__version__)
print("numpy: ", np.__version__)

xftsim: 0.3.0.dev110
numpy:  1.26.4


## 1. The smallest useful simulation — single trait, DGE + E

Every architecture is a string in the formula DSL plus an `effects` dict.
The simplest possible model is one trait with a direct genetic component
and additive environmental noise:

```
Y.G ~ genetic(eff)
Y.E ~ noise(var_E)
Y   ~ Y.G + Y.E
```

`Y.G` and `Y.E` are stored as separate columns in the per-generation
phenotype dict, so the variance partition is recoverable by name. `Y` is
just an arithmetic aggregation node.

For uniform-AF founders with MAF drawn from `U(0.1, 0.9)`, picking
`var_E = 1 - h²` makes `var(Y) ≈ 1`.

In [2]:
from xftsim.founders import founder_haplotypes_uniform_AFs
from xftsim.neffect import AdditiveEffects
from xftsim.narch import Architecture
from xftsim.nmate import RandomMating
from xftsim.reproduce import RecombinationMap
from xftsim.nsim import NSimulation

np.random.seed(0)

n_fam = 4_000
m     = 2_000
h2    = 0.5

founder_hap = founder_haplotypes_uniform_AFs(n=2 * n_fam, m=m)
eff = AdditiveEffects.from_h2(h2=h2, m=m, seed=1)

arch = Architecture(formula="""
Y.G ~ genetic(eff)
Y.E ~ noise({varE})
Y   ~ Y.G + Y.E
""".format(varE=1 - h2),
    effects={"eff": eff},
)

sim = NSimulation(
    founder_haplotypes=founder_hap,
    architecture=arch,
    mating_regime=RandomMating(offspring_per_pair=2),
    recombination_map=RecombinationMap.constant_map(m=m, p=0.5),
    seed=1,
)
sim.run(2)   # gen 0 (founders) + gen 1 (offspring)

ph = sim.phenotype_history[1]
print(f"var(Y.G) = {ph['Y.G'].var():.3f}   target {h2}")
print(f"var(Y.E) = {ph['Y.E'].var():.3f}   target {1-h2}")
print(f"var(Y)   = {ph['Y'].var():.3f}     target ~1")

var(Y.G) = 0.516   target 0.5
var(Y.E) = 0.514   target 0.5
var(Y)   = 1.040     target ~1


**`genetic(eff)`** uses `standardized_matvec` internally — it
centers and scales each column to unit variance under HWE before
multiplying by the per-allele effects. So `AdditiveEffects.from_h2(h2=...,
m=...)` draws `β ~ N(0, h²/m)` and the resulting genetic value variance
hits `h²` directly. Good for quick experiments.

We'll switch to the **raw-dosage** path (`haplotypeGenetic`) in §3 — it
gives us per-haplotype scores but at the cost of having to rescale
`β` by `1/sqrt(2p(1-p))` ourselves.

## 2. Two traits with correlated direct effects

To put a target genetic correlation `ρ_g` between the two traits, draw
the per-SNP `β` matrix from a bivariate normal with off-diagonal
correlation `ρ_g`:

`β ~ N(0, Σ/m)` with `Σ = [[h²_A, ρ_g·sqrt(h²_A·h²_B)], [ρ_g·sqrt(h²_A·h²_B), h²_B]]`

Then register one `AdditiveEffects` per column. The per-trait
heritabilities still hit `h²_A` and `h²_B`, and the cross-trait DGE
covariance carries through.

In [3]:
def draw_bivariate_betas(h2_A, h2_B, rho_g, m, seed):
    rng = np.random.default_rng(seed)
    Sigma = np.array([
        [h2_A,                      rho_g * np.sqrt(h2_A * h2_B)],
        [rho_g * np.sqrt(h2_A*h2_B), h2_B],
    ]) / m
    return rng.multivariate_normal(mean=[0.0, 0.0], cov=Sigma, size=m)

h2_A, h2_B, rho_g = 0.5, 0.5, 0.3
betas = draw_bivariate_betas(h2_A, h2_B, rho_g, m=m, seed=2)

eff_A = AdditiveEffects.from_array(betas[:, 0])  # standardized convention
eff_B = AdditiveEffects.from_array(betas[:, 1])

arch = Architecture(formula="""
TraitA.G ~ genetic(eff_A)
TraitA.E ~ noise({eA})
TraitA   ~ TraitA.G + TraitA.E

TraitB.G ~ genetic(eff_B)
TraitB.E ~ noise({eB})
TraitB   ~ TraitB.G + TraitB.E
""".format(eA=1-h2_A, eB=1-h2_B),
    effects={"eff_A": eff_A, "eff_B": eff_B},
)

founder_hap = founder_haplotypes_uniform_AFs(n=2*n_fam, m=m)
sim = NSimulation(
    founder_haplotypes=founder_hap,
    architecture=arch,
    mating_regime=RandomMating(offspring_per_pair=2),
    recombination_map=RecombinationMap.constant_map(m=m, p=0.5),
    seed=2,
)
sim.run(2)

ph = sim.phenotype_history[1]
gA, gB = ph["TraitA.G"], ph["TraitB.G"]
print(f"var(TraitA.G) = {gA.var():.3f}   target {h2_A}")
print(f"var(TraitB.G) = {gB.var():.3f}   target {h2_B}")
print(f"cor(A.G, B.G) = {np.corrcoef(gA, gB)[0,1]:+.3f}   target {rho_g}")

var(TraitA.G) = 0.515   target 0.5
var(TraitB.G) = 0.511   target 0.5
cor(A.G, B.G) = +0.317   target 0.3


## 3. Splitting the direct genetic score into transmitted halves

For indirect-effect work, we need to know which alleles a child got from
each parent. `haplotypeGenetic(eff, haplotype='maternal')` projects only
the maternal-side haplotype copy onto `eff` — and likewise for paternal.
Their sum is the diploid DGE.

**One subtlety:** unlike `genetic(...)`, `haplotypeGenetic(...)` operates
on raw 0/1 haplotype dosages — no per-SNP standardization. To target a
heritability of `h²` we must rescale `β` by `1/sqrt(2p(1-p))` per SNP so
that `Σ 2p(1-p)·β² ≈ h²`. Otherwise variance comes out at `h²/2` for
uniform MAF=0.5.

In [4]:
# Get founder allele frequencies for the rescale
afs = np.asarray(founder_hap.variants.af, dtype=np.float64)
betas_raw = betas / np.sqrt(2 * afs * (1 - afs))[:, None]

# Tell AdditiveEffects we're now in raw-dosage units
eff_A = AdditiveEffects.from_array(betas_raw[:, 0], standardized=False)
eff_B = AdditiveEffects.from_array(betas_raw[:, 1], standardized=False)

arch = Architecture(formula="""
TraitA.T_mat ~ haplotypeGenetic(eff_A, haplotype='maternal')
TraitA.T_pat ~ haplotypeGenetic(eff_A, haplotype='paternal')
TraitA.DGE   ~ TraitA.T_mat + TraitA.T_pat
TraitA.E     ~ noise({eA})
TraitA       ~ TraitA.DGE + TraitA.E

TraitB.T_mat ~ haplotypeGenetic(eff_B, haplotype='maternal')
TraitB.T_pat ~ haplotypeGenetic(eff_B, haplotype='paternal')
TraitB.DGE   ~ TraitB.T_mat + TraitB.T_pat
TraitB.E     ~ noise({eB})
TraitB       ~ TraitB.DGE + TraitB.E
""".format(eA=1-h2_A, eB=1-h2_B),
    effects={"eff_A": eff_A, "eff_B": eff_B},
)

sim = NSimulation(
    founder_haplotypes=founder_hap,
    architecture=arch,
    mating_regime=RandomMating(offspring_per_pair=2),
    recombination_map=RecombinationMap.constant_map(m=m, p=0.5),
    seed=3,
)
sim.run(2)

ph = sim.phenotype_history[1]
recon = ph["TraitB.T_mat"] + ph["TraitB.T_pat"]
print(f"var(TraitB.DGE)        = {ph['TraitB.DGE'].var():.3f}   target {h2_B}")
print(f"var(TraitB.T_mat)      = {ph['TraitB.T_mat'].var():.3f}   target {h2_B/2}")
print(f"var(TraitB.T_pat)      = {ph['TraitB.T_pat'].var():.3f}   target {h2_B/2}")
print(f"cov(T_mat, T_pat)      = {np.cov(ph['TraitB.T_mat'], ph['TraitB.T_pat'])[0,1]:+.3f}   target ~0")
print(f"max|T_mat+T_pat - DGE| = {np.abs(recon - ph['TraitB.DGE']).max():.2e}   must be ~0")

var(TraitB.DGE)        = 0.515   target 0.5
var(TraitB.T_mat)      = 0.259   target 0.25
var(TraitB.T_pat)      = 0.254   target 0.25
cov(T_mat, T_pat)      = +0.001   target ~0
max|T_mat+T_pat - DGE| = 0.00e+00   must be ~0


The two transmitted-allele components are uncorrelated under
random mating (each haplotype is an independent meiotic draw) and sum
exactly to the diploid DGE. We're now ready to source the indirect
channel from non-transmitted alleles.

## 4. Vertical (indirect genetic) transmission with T/NT separation

The biological pathway for indirect genetic effects is
`parent_genome → parent_phenotype → child_environment → child_phenotype`.
The parent's *full* DGE drives the parent's phenotype, so the natural
input to the child's IGE channel is the parent's full DGE, not just the
non-transmitted half.

But the **identification problem** is that half of the parent's DGE
*is* in the child's own genome (the transmitted alleles), which makes
the child's DGE and child's IGE positively correlated. The classical
NTC fix is to source identification — separately — from the
**non-transmitted** half: `NT = parental_DGE − transmitted`.

So we wire two parental lookups:

* **`mother(TraitB.DGE, normalize=False, founder=...)`** → raw parental
  DGE on the same scale as `TraitB.T_mat`. Subtraction gives `NT_m`
  exactly: `NT_m = PDGE_m − T_mat`. Used for downstream estimation.
* **`mother(TraitB.DGE, normalize=True, founder=...)`** → standardized
  parental DGE every generation. This feeds the IGE channel with
  `var(IGE) ≡ B_vIGE` regardless of drift or AM-induced variance shifts.

Coefficients are `sqrt(B_vIGE · p_mom)` and `sqrt(B_vIGE · p_dad)`. Below
we set `p_mom = p_dad = 0.5` so each parent contributes half.

In [5]:
B_vIGE = 0.4
p_mom, p_dad = 0.5, 0.5

formula = """\
TraitB.T_mat ~ haplotypeGenetic(eff_B, haplotype='maternal')
TraitB.T_pat ~ haplotypeGenetic(eff_B, haplotype='paternal')
TraitB.DGE   ~ TraitB.T_mat + TraitB.T_pat

# Raw parental DGE (lookup) — used for T/NT subtraction
TraitB.PDGE_m ~ mother(TraitB.DGE, normalize=False, founder=noise({h2B}))
TraitB.PDGE_f ~ father(TraitB.DGE, normalize=False, founder=noise({h2B}))
TraitB.NT_m   ~ TraitB.PDGE_m - TraitB.T_mat
TraitB.NT_f   ~ TraitB.PDGE_f - TraitB.T_pat

# Variance-pinned IGE channel
TraitB.IGE_src_m ~ mother(TraitB.DGE, normalize=True, founder=noise(1.0))
TraitB.IGE_src_f ~ father(TraitB.DGE, normalize=True, founder=noise(1.0))
TraitB.IGE       ~ {c_m} * TraitB.IGE_src_m + {c_f} * TraitB.IGE_src_f

TraitB.E ~ noise({eB})
TraitB   ~ TraitB.DGE + TraitB.IGE + TraitB.E
""".format(
    h2B=h2_B,
    c_m=np.sqrt(B_vIGE * p_mom),
    c_f=np.sqrt(B_vIGE * p_dad),
    eB=1 - h2_B - B_vIGE,
)

arch = Architecture(formula=formula, effects={"eff_B": eff_B})
sim = NSimulation(
    founder_haplotypes=founder_hap,
    architecture=arch,
    mating_regime=RandomMating(offspring_per_pair=2),
    recombination_map=RecombinationMap.constant_map(m=m, p=0.5),
    seed=4,
)
sim.run(2)

ph = sim.phenotype_history[1]
nt_recon = ph["TraitB.PDGE_m"] - ph["TraitB.T_mat"]
print(f"var(TraitB.DGE)            = {ph['TraitB.DGE'].var():.3f}   target {h2_B}")
print(f"var(TraitB.IGE)            = {ph['TraitB.IGE'].var():.3f}   target {B_vIGE}  (pinned)")
print(f"var(TraitB.E)              = {ph['TraitB.E'].var():.3f}   target {1-h2_B-B_vIGE}")
print(f"var(TraitB.NT_m)           = {ph['TraitB.NT_m'].var():.3f}   target {h2_B/2}")
print(f"max|PDGE_m - T_mat - NT_m| = {np.abs(nt_recon - ph['TraitB.NT_m']).max():.2e}   must be ~0")
cov_dge_ige = np.cov(ph["TraitB.DGE"], ph["TraitB.IGE"])[0, 1]
print(f"cov(DGE, IGE)              = {cov_dge_ige:+.3f}   theoretical ~{(0.5*h2_B*B_vIGE)**0.5:+.3f}")
print(f"var(TraitB)                = {ph['TraitB'].var():.3f}   inflated by 2·cov(DGE, IGE) — see below")

var(TraitB.DGE)            = 0.511   target 0.5
var(TraitB.IGE)            = 0.402   target 0.4  (pinned)
var(TraitB.E)              = 0.096   target 0.09999999999999998
var(TraitB.NT_m)           = 0.262   target 0.25
max|PDGE_m - T_mat - NT_m| = 0.00e+00   must be ~0
cov(DGE, IGE)              = +0.323   theoretical ~+0.316
var(TraitB)                = 1.659   inflated by 2·cov(DGE, IGE) — see below


The total `var(TraitB)` is genuinely above 1 — that's the
biological DGE↔IGE confound at work. Without the T/NT split it would be
unrecoverable. With it, we can identify the indirect channel cleanly,
which is the next section.

## 5. Estimating direct and indirect coefficients

Under the model in §4:

```
child_TraitB = (T_mat + T_pat)
             + α_m · (T_mat + NT_m)           # IGE_src_m ≈ (T_mat + NT_m) up to scale
             + α_f · (T_pat + NT_f)
             + E
           = T_mat·(1+α_m) + T_pat·(1+α_f) + α_m·NT_m + α_f·NT_f + E
```

where `α_m ≈ sqrt(B_vIGE · p_mom / var(parental_DGE))` is the realized
coefficient on the parent's *raw* DGE in the IGE channel after the
internal normalize-by-SD step. So an OLS regression of child phenotype
on the four columns recovers:

* `β(T_mat) = 1 + α_m`,  `β(T_pat) = 1 + α_f`
* `β(NT_m) = α_m`,        `β(NT_f) = α_f`

Subtracting the NT coefficient from the matching T coefficient yields
the **direct effect of one allele's worth of polygenic score on the
child's phenotype** (~1 by construction — we built the model so that a
unit of parental DGE adds a unit to child DGE, plus its IGE pathway).

We re-run the §4 model at a larger sample size for tight estimates, then
fit the regression.

In [6]:
# Re-run with a bigger n for precision
np.random.seed(10)
n_fam_big = 10_000
m_big     = 2_000

founder_big = founder_haplotypes_uniform_AFs(n=2 * n_fam_big, m=m_big)
afs_big = np.asarray(founder_big.variants.af, dtype=np.float64)

beta_big = draw_bivariate_betas(h2_A, h2_B, rho_g, m=m_big, seed=11)
beta_big_raw = beta_big / np.sqrt(2 * afs_big * (1 - afs_big))[:, None]
eff_B_big = AdditiveEffects.from_array(beta_big_raw[:, 1], standardized=False)

arch_big = Architecture(formula=formula, effects={"eff_B": eff_B_big})
sim_big = NSimulation(
    founder_haplotypes=founder_big,
    architecture=arch_big,
    mating_regime=RandomMating(offspring_per_pair=2),
    recombination_map=RecombinationMap.constant_map(m=m_big, p=0.5),
    seed=12,
)
sim_big.run(2)
ph = sim_big.phenotype_history[1]

# Build design matrix
X = np.column_stack([
    ph["TraitB.T_mat"], ph["TraitB.T_pat"],
    ph["TraitB.NT_m"],  ph["TraitB.NT_f"],
])
y = ph["TraitB"]
ols = sm.OLS(y, sm.add_constant(X)).fit()

names = ["intercept", "T_mat", "T_pat", "NT_m", "NT_f"]
out = pd.DataFrame({
    "estimate":   ols.params,
    "std_error":  ols.bse,
    "t_value":    ols.tvalues,
    "p_value":    ols.pvalues,
}, index=names)
out

,estimate,std_error,t_value,p_value
intercept,1.278768,0.005012,255.161833,0.0
T_mat,1.631614,0.004459,365.940428,0.0
T_pat,1.637219,0.004441,368.636537,0.0
NT_m,0.634530,0.004511,140.652121,0.0
NT_f,0.620610,0.004475,138.679488,0.0


Compare against the theoretical targets. We seeded the IGE
channel with coefficient `sqrt(B_vIGE · p_parent)` on a *standardized*
parental DGE, so the realized coefficient on the raw parental DGE is
`α = sqrt(B_vIGE · p_parent / var(parental_DGE))`.

In [7]:
parent_dge_var = sim_big.phenotype_history[0]["TraitB.DGE"].var()
alpha_m = np.sqrt(B_vIGE * p_mom / parent_dge_var)
alpha_f = np.sqrt(B_vIGE * p_dad / parent_dge_var)

theory = pd.DataFrame({
    "theoretical": [np.nan, 1 + alpha_m, 1 + alpha_f, alpha_m, alpha_f],
    "estimate":    ols.params,
}, index=names)
theory["error"] = theory.estimate - theory.theoretical
theory

,theoretical,estimate,error
intercept,NaN,1.278768,NaN
T_mat,1.630436,1.631614,0.001178
T_pat,1.630436,1.637219,0.006783
NT_m,0.630436,0.634530,0.004094
NT_f,0.630436,0.620610,-0.009827


**Direct effect** (per allele unit of polygenic score) is recovered as
`β(T_mat) − β(NT_m)` and `β(T_pat) − β(NT_f)`, both of which should sit
at `1.0`. **Indirect effect** is `β(NT_m)` and `β(NT_f)`, which match
`α_m` and `α_f` directly.

Notice that the simple regression of child phenotype on its own DGE
*without* NT controls would over-attribute the indirect channel to the
direct channel — the classical "dynastic confound." Throw it in for
contrast:

In [8]:
contam = sm.OLS(y, sm.add_constant(ph["TraitB.DGE"])).fit()
print(f"naive direct estimate = {contam.params[1]:.3f}  "
      f"(should be ~{1 + alpha_m:.3f}, conflating direct + indirect)")

naive direct estimate = 1.641  (should be ~1.630, conflating direct + indirect)


## 6. Layering on assortative mating

`xftsim.nmate.LinearAssortativeMating` rank-pairs each sex on a
phenotypic composite (sum of standardized components from
`component_names`). The `r` argument is the **target per-cell exchangeable
cross-mate correlation** under K traits.

* **K = 1**: the spousal correlation on that single trait is `≈ r`.
* **K > 1**: every (parent1_t_i, parent2_t_j) cell — both diagonal
  (per-trait) and off-diagonal (cross-trait) — is targeted at `r`.
  Feasibility bound is `|r| ≤ 1/K`. Beyond that the latent score
  correlation `R = K·r` saturates and you get less assortment than you
  asked for.

The current implementation **warns** at construction when `|r| > 1/K`,
and again at runtime when the realized `R` exceeds 1. Below: each of the
three regimes, side by side.

In [9]:
from xftsim.nmate import LinearAssortativeMating
import warnings

def run_am(mating, label):
    sim = NSimulation(
        founder_haplotypes=founder_haplotypes_uniform_AFs(n=2*n_fam, m=m),
        architecture=Architecture(
            formula="""\
TraitA.G ~ genetic(eff_A)
TraitA.E ~ noise({eA})
TraitA   ~ TraitA.G + TraitA.E
TraitB.G ~ genetic(eff_B)
TraitB.E ~ noise({eB})
TraitB   ~ TraitB.G + TraitB.E
""".format(eA=1-h2_A, eB=1-h2_B),
            effects={"eff_A": AdditiveEffects.from_h2(h2_A, m=m, seed=21),
                     "eff_B": AdditiveEffects.from_h2(h2_B, m=m, seed=22)},
        ),
        mating_regime=mating,
        recombination_map=RecombinationMap.constant_map(m=m, p=0.5),
        seed=30,
    )
    sim.run(2)
    ph = sim.phenotype_history[1]
    ped = sim.pedigree_history[1]
    par = sim.phenotype_history[0]
    moms = par["TraitA"][ped.maternal_idx], par["TraitB"][ped.maternal_idx]
    dads = par["TraitA"][ped.paternal_idx], par["TraitB"][ped.paternal_idx]
    rAA = np.corrcoef(moms[0], dads[0])[0, 1]
    rBB = np.corrcoef(moms[1], dads[1])[0, 1]
    rAB = np.corrcoef(moms[0], dads[1])[0, 1]
    rBA = np.corrcoef(moms[1], dads[0])[0, 1]
    print(f"{label:35s} cor(A,A)={rAA:+.3f}  cor(B,B)={rBB:+.3f}  "
          f"cor(A,B)={rAB:+.3f}  cor(B,A)={rBA:+.3f}")

# 1) random mating — baseline
run_am(RandomMating(offspring_per_pair=2),
       "random")

# 2) single-trait AM on TraitA
run_am(LinearAssortativeMating(component_names=["TraitA"], r=0.3,
                                offspring_per_pair=2),
       "single-trait r=0.3 on TraitA")

# 3) cross-trait AM — feasible (|r| <= 1/K = 0.5)
run_am(LinearAssortativeMating(component_names=["TraitA", "TraitB"], r=0.3,
                                offspring_per_pair=2),
       "cross-trait r=0.3 (K=2, feasible)")

# 4) cross-trait AM — infeasible: should warn at construction AND at runtime
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    run_am(LinearAssortativeMating(component_names=["TraitA", "TraitB"], r=0.75,
                                    offspring_per_pair=2),
           "cross-trait r=0.75 (K=2, INFEASIBLE)")
    for w in caught:
        if "LinearAssortativeMating" in str(w.message):
            print(f"  warn: {str(w.message)[:120]}…")

random                              cor(A,A)=+0.012  cor(B,B)=-0.012  cor(A,B)=-0.007  cor(B,A)=+0.014


single-trait r=0.3 on TraitA        cor(A,A)=+0.296  cor(B,B)=-0.016  cor(A,B)=+0.000  cor(B,A)=+0.010


cross-trait r=0.3 (K=2, feasible)   cor(A,A)=+0.304  cor(B,B)=+0.306  cor(A,B)=+0.290  cor(B,A)=+0.297


cross-trait r=0.75 (K=2, INFEASIBLE) cor(A,A)=+0.504  cor(B,B)=+0.504  cor(A,B)=+0.515  cor(B,A)=+0.500
  warn: LinearAssortativeMating: |r|=0.75 exceeds 1/K=0.5 for K=2 traits. Under uncorrelated within-person traits this target is…
  warn: LinearAssortativeMating: realized latent score correlation R=1.48 exceeds 1 for r=0.75, K=2. Clamping to 0.999 (assortme…


**Reading the matrix:** under exchangeable cross-trait AM with
`r = 0.3` you'd want all four (A,A), (B,B), (A,B), (B,A) parental
correlations close to 0.3. With `r = 0.75` they all top out around 0.5
because the latent score correlation got clamped at saturation.

**Putting it all together:** to reproduce a study of indirect genetic
effects under assortment, swap §6's mating regime into §4's architecture
and re-run §5's NTC regression. Under AM, `var(parent_DGE)` inflates each
generation, so `α_m`, `α_f` will drift from their gen-0 values — but
they remain identifiable from the OLS fit, generation by generation.